In [1]:
library("xgboost")
library("Matrix")
library('Ckmeans.1d.dp')
library('lightgbm')



载入程辑包：‘lightgbm’


The following object is masked from ‘package:xgboost’:

    slice




## Read data and process data labels

In [2]:
time_matrix <- matrix(0,ncol = 3, nrow =4)
colnames(time_matrix) <- c("user_time", "system_time", "elapsed_time")
start_time = Sys.time()

In [3]:
data=read.csv('Monthly_features_matrix.csv')
data=data[,2:dim(data)[2]]

In [4]:
dim(data)

[1] 48000    27

In [5]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,seasonal_strength,⋯,diff2_acf1,diff2_acf10,seas_acf1,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,12,1,12,0.9472661,1.912037e-08,0.872829,10.9548535,0.18758224,0.1214246,0.8404932,⋯,-0.3962824,0.4604731,0.8293577,469,0.05674529,0.05674529,0.05674529,0,0,0
2,12,1,12,0.8085632,2.203382e-07,0.943755,4.7359892,0.09830677,0.1405957,0.4180888,⋯,-0.5668803,0.3576969,0.5860238,469,0.05383134,0.05383134,0.05383134,0,0,0
3,12,1,12,0.9992200,8.446236e-12,21.172689,4.0198303,0.36279469,0.4204365,0.5104123,⋯,-0.4913136,0.3353221,0.9154861,469,0.06982851,0.06982851,0.06982851,0,0,0
4,12,1,12,0.8687883,3.833819e-06,-7.295270,0.3739581,0.50800514,0.5935817,0.2357374,⋯,-0.5149001,0.3611336,0.3110420,82,0.06202483,0.06202483,0.06202483,0,0,0
5,12,1,12,0.7457338,1.006626e-05,4.869531,0.5621160,0.19177799,0.2581481,0.3207559,⋯,-0.6516841,0.5925659,0.1199568,112,0.04698277,0.04698277,0.04698277,0,0,0
6,12,1,12,0.9976576,3.100646e-10,13.985335,-12.7683661,0.50582153,0.2947603,0.4390163,⋯,-0.4473514,0.2338179,0.9367465,666,0.07257485,0.07257485,0.07257485,0,0,0


In [6]:
dlist= load('Monthly_nnetar_datalist.RData')
datalist=eval(parse(text = dlist ))
res=datalist[[1]]
MASE=res[,,,6]
m=5

In [7]:
dim(MASE)

[1] 48000     5     4

In [8]:
whichmin<-function(x){
    minx=min(x[x>0])
    loc=which(x==minx)[1]-1
    loc
}

meanunique=function(x)
    {
    mean(unique(x))
}

In [9]:
nanum=c()
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    a=apply(count,1,min)
    if(sum(is.na(a))>0)
        {
        nanum=append(nanum,i)
    }
    }

In [10]:
nanum

[1]     4     5     7     8    41    42    43    56    58    59    61    62
   [13]    63    64    65    66    67    68    69    72    73    74    75    78
   [25]    79    81    82    83    84    85    86    87    88    91   103   104
   [37]   106   107   108   109   110   111   112   113   131   132   133   134
   [49]   135   136   137   141   143   147   153   154   155   160   161   162
   [61]   165   167   170   173   191   192   194   195   203   204   205   207
   [73]   215   216   217   218   219   220   221   222   223   226   227   228
   [85]   229   230   231   232   236   238   239   240   241   242   246   247
   [97]   248   252   253   254   255   256   257   258   259   261   262   263
  [109]   264   265   266   271   278   280   282   283   284   286   287   289
  [121]   290   292   293   295   296   297   298   300   303   313   314   315
  [133]   317   318   323   325   326   327   341   344   345   346   347   348
  [145]   349   354   365   366   373   384   388   389   391   396   398   399
  [157]   400   401   402   405   410   411   412   414   420   423   424   426
  [169]   427   434   435   436   439   440   441   447   460   461   462   463
  [181]   469   470   471   472   475   476   477   480   484   503   516   517
  [193]   544   564   566   574   575   576   577   586   591   594   610   612
  [205]   613   634   642   643   645   692   722   737   738   743   745   760
  [217]   765   773   774   777   782   790   800   801   808   843   845   875
  [229]   910   926   953   954   955   958  1064  1087  1112  1224  1232  1271
  [241]  1272  1273  1305  1352  1374  1379  1424  1425  1485  1512  1642  1665
  [253]  1684  1685  1723  1724  1729  1746  1768  1777  1781  1846  1849  1851
  [265]  1860  1861  1862  1913  1914  1949  1950  2040  2077  2130  2136  2139
  [277]  2140  2144  2147  2148  2149  2150  2151  2191  2192  2193  2196  2232
  [289]  2233  2267  2361  2373  2380  2383  2393  2404  2406  2416  2417  2418
  [301]  2419  2421  2462  2508  2509  2511  2582  2586  2587  2588  2589  2590
  [313]  2591  2592  2593  2594  2627  2628  2629  2643  2645  2648  2649  2650
  [325]  2674  2675  2678  2679  2680  2682  2685  2712  2721  2740  2741  2742
  [337]  2743  2744  2745  2746  2747  2748  2749  2750  2751  2752  2753  2754
  [349]  2755  2756  2757  2758  2759  2760  2761  2762  2763  2764  2765  2766
  [361]  2767  2768  2769  2770  2771  2772  2773  2774  2775  2776  2777  2778
  [373]  2779  2780  2781  2782  2783  2784  2785  2786  2787  2788  2789  2790
  [385]  2791  2792  2793  2794  2795  2796  2797  2798  2799  2800  2801  2802
  [397]  2803  2804  2805  2806  2836  2837  2838  2839  2840  2841  2842  2843
  [409]  2844  2845  2846  2847  2848  2849  2850  2851  2866  2867  2900  2901
  [421]  2911  2912  2913  2914  2915  2928  2952  2954  2955  2956  2964  2966
  [433]  2967  2968  2969  2986  2992  3006  3007  3011  3013  3014  3035  3036
  [445]  3037  3047  3050  3052  3053  3054  3069  3071  3072  3073  3074  3075
  [457]  3076  3078  3080  3081  3084  3116  3119  3139  3140  3141  3142  3148
  [469]  3161  3162  3163  3164  3165  3166  3198  3212  3213  3237  3267  3268
  [481]  3269  3270  3284  3329  3380  3385  3387  3390  3392  3466  3467  3468
  [493]  3469  3470  3542  3543  3544  3545  3546  3547  3548  3549  3550  3551
  [505]  3552  3553  3554  3555  3556  3557  3603  3614  3616  3619  3802  3891
  [517]  3903  3910  3911  3912  3913  3914  3915  3925  3936  3940  3948  4080
  [529]  4081  4082  4083  4084  4085  4086  4087  4088  4089  4090  4091  4092
  [541]  4093  4098  4100  4102  4103  4105  4108  4109  4110  4111  4112  4113
  [553]  4114  4115  4116  4117  4118  4120  4121  4122  4123  4124  4125  4126
  [565]  4127  4128  4129  4130  4131  4132  4133  4134  4136  4137  4138  4139
  [577]  4140  4141  4142  4143  4144  4145  4146  4147  4148  4149  4223  4224
  [589]  4225  4226  4230  4371  4427  4428  4429  4430  4431  4432  4433  4434
  [6

In [11]:
realbestmin=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    min_value=apply(count,1,min)
    if (max(min_value,na.rm = TRUE)==0){
        realbestmin[i,]=0
        }
    else
        {
        min_value[is.na(min_value)]=100 
        realbestmin[i,]= whichmin(min_value)
    }
    }

In [12]:
table(realbestmin)

realbestmin
    0     1     2     3     4 
10824  8212  7894 11816  9254 

In [13]:
realbestmean=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    mean_value=apply(count,1,meanunique)
    if (max(mean_value,na.rm = TRUE)==0){
        realbestmean[i,]=0
        }
    else
        {
        mean_value[is.na(mean_value)]=100 
        realbestmean[i,]= whichmin(mean_value)
    }
    }

In [14]:
realbestmean

4
3
3
1
3
1
1
2
0
4
0


In [15]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,seasonal_strength,⋯,diff2_acf1,diff2_acf10,seas_acf1,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,12,1,12,0.9472661,1.912037e-08,0.872829,10.9548535,0.18758224,0.1214246,0.8404932,⋯,-0.3962824,0.4604731,0.8293577,469,0.05674529,0.05674529,0.05674529,0,0,0
2,12,1,12,0.8085632,2.203382e-07,0.943755,4.7359892,0.09830677,0.1405957,0.4180888,⋯,-0.5668803,0.3576969,0.5860238,469,0.05383134,0.05383134,0.05383134,0,0,0
3,12,1,12,0.9992200,8.446236e-12,21.172689,4.0198303,0.36279469,0.4204365,0.5104123,⋯,-0.4913136,0.3353221,0.9154861,469,0.06982851,0.06982851,0.06982851,0,0,0
4,12,1,12,0.8687883,3.833819e-06,-7.295270,0.3739581,0.50800514,0.5935817,0.2357374,⋯,-0.5149001,0.3611336,0.3110420,82,0.06202483,0.06202483,0.06202483,0,0,0
5,12,1,12,0.7457338,1.006626e-05,4.869531,0.5621160,0.19177799,0.2581481,0.3207559,⋯,-0.6516841,0.5925659,0.1199568,112,0.04698277,0.04698277,0.04698277,0,0,0
6,12,1,12,0.9976576,3.100646e-10,13.985335,-12.7683661,0.50582153,0.2947603,0.4390163,⋯,-0.4473514,0.2338179,0.9367465,666,0.07257485,0.07257485,0.07257485,0,0,0


In [16]:
set.seed(100)
index = sample(2,nrow(data),replace = TRUE,prob=c(0.7,0.3))

In [17]:
train_data=data[index==1,]
test_data=data[index==2,]
train_label_min=realbestmin[index==1,]
test_label_min=realbestmin[index==2,]
train_label_mean=realbestmean[index==1,]
test_label_mean=realbestmean[index==2,]

In [18]:
head(train_data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,seasonal_strength,⋯,diff2_acf1,diff2_acf10,seas_acf1,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,12,1,12,0.9472661,1.912037e-08,0.872829,10.9548535,0.18758224,0.1214246,0.8404932,⋯,-0.3962824,0.4604731,0.8293577,469,0.05674529,0.05674529,0.05674529,0,0,0
2,12,1,12,0.8085632,2.203382e-07,0.943755,4.7359892,0.09830677,0.1405957,0.4180888,⋯,-0.5668803,0.3576969,0.5860238,469,0.05383134,0.05383134,0.05383134,0,0,0
3,12,1,12,0.9992200,8.446236e-12,21.172689,4.0198303,0.36279469,0.4204365,0.5104123,⋯,-0.4913136,0.3353221,0.9154861,469,0.06982851,0.06982851,0.06982851,0,0,0
4,12,1,12,0.8687883,3.833819e-06,-7.295270,0.3739581,0.50800514,0.5935817,0.2357374,⋯,-0.5149001,0.3611336,0.3110420,82,0.06202483,0.06202483,0.06202483,0,0,0
5,12,1,12,0.7457338,1.006626e-05,4.869531,0.5621160,0.19177799,0.2581481,0.3207559,⋯,-0.6516841,0.5925659,0.1199568,112,0.04698277,0.04698277,0.04698277,0,0,0
6,12,1,12,0.9976576,3.100646e-10,13.985335,-12.7683661,0.50582153,0.2947603,0.4390163,⋯,-0.4473514,0.2338179,0.9367465,666,0.07257485,0.07257485,0.07257485,0,0,0


In [19]:
end_time = Sys.time()

In [20]:
time_matrix[1,]=end_time-start_time

In [21]:
end_time-start_time

Time difference of 19.28747 secs

## Target the interval where the actual error is minimum

In [22]:
start_time = Sys.time()

In [23]:
dtrain_xg_min_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_min)) 
dtrain_xg_min_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)) )

In [24]:
dtrain_lg_min_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_min))
dtrain_lg_min_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)))

In [25]:
xgb_min_reg <- xgboost(data = dtrain_xg_min_reg, nround=100)

[1]	train-rmse:1.717546 
[2]	train-rmse:1.494097 
[3]	train-rmse:1.367672 
[4]	train-rmse:1.297307 
[5]	train-rmse:1.258110 
[6]	train-rmse:1.234723 
[7]	train-rmse:1.219706 
[8]	train-rmse:1.208295 
[9]	train-rmse:1.200432 
[10]	train-rmse:1.193467 
[11]	train-rmse:1.189482 
[12]	train-rmse:1.186445 
[13]	train-rmse:1.185284 
[14]	train-rmse:1.183805 
[15]	train-rmse:1.180842 
[16]	train-rmse:1.177202 
[17]	train-rmse:1.175663 
[18]	train-rmse:1.172185 
[19]	train-rmse:1.170842 
[20]	train-rmse:1.165507 
[21]	train-rmse:1.163062 
[22]	train-rmse:1.160947 
[23]	train-rmse:1.158142 
[24]	train-rmse:1.153431 
[25]	train-rmse:1.152216 
[26]	train-rmse:1.148526 
[27]	train-rmse:1.145918 
[28]	train-rmse:1.142743 
[29]	train-rmse:1.139650 
[30]	train-rmse:1.139272 
[31]	train-rmse:1.136147 
[32]	train-rmse:1.132675 
[33]	train-rmse:1.132215 
[34]	train-rmse:1.128989 
[35]	train-rmse:1.126119 
[36]	train-rmse:1.124130 
[37]	train-rmse:1.121234 
[38]	train-rmse:1.117761 
[39]	train-rmse:1.115

In [26]:
xgb_min_cl <- xgboost(data = dtrain_xg_min_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-mlogloss:1.522099 
[2]	train-mlogloss:1.464419 
[3]	train-mlogloss:1.421188 
[4]	train-mlogloss:1.388207 
[5]	train-mlogloss:1.361329 
[6]	train-mlogloss:1.339267 
[7]	train-mlogloss:1.320911 
[8]	train-mlogloss:1.305706 
[9]	train-mlogloss:1.291235 
[10]	train-mlogloss:1.279405 
[11]	train-mlogloss:1.267428 
[12]	train-mlogloss:1.257418 
[13]	train-mlogloss:1.248994 
[14]	train-mlogloss:1.241427 
[15]	train-mlogloss:1.234098 
[16]	train-mlogloss:1.227503 
[17]	train-mlogloss:1.219976 
[18]	train-mlogloss:1.214410 
[19]	train-mlogloss:1.210445 
[20]	train-mlogloss:1.205435 
[21]	train-mlogloss:1.200466 
[22]	train-mlogloss:1.194654 
[23]	train-mlogloss:1.189775 
[24]	train-mlogloss:1.187514 
[25]	train-mlogloss:1.185359 
[26]	train-mlogloss:1.180508 
[27]	train-mlogloss:1.177089 
[28]	train-mlogloss:1.172830 
[29]	train-mlogloss:1.169591 
[30]	train-mlogloss:1.165501 
[31]	train-mlogloss:1.161478 
[32]	train-mlogloss:1.158127 
[33]	train-mlogloss:1.154914 
[34]	train-mlogloss

In [27]:
lgb_min_reg <- lgb.train(data = dtrain_lg_min_reg, nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011917 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4868
[LightGBM] [Info] Number of data points in the train set: 33791, number of used features: 21
[LightGBM] [Info] Start training from score 2.013820


In [28]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_min_cl <- lgb.train(data = dtrain_lg_min_cl,nrounds = 100,params=params)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006550 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4868
[LightGBM] [Info] Number of data points in the train set: 33791, number of used features: 21
[LightGBM] [Info] Start training from score -1.485881
[LightGBM] [Info] Start training from score -1.780430
[LightGBM] [Info] Start training from score -1.801723
[LightGBM] [Info] Start training from score -1.405506
[LightGBM] [Info] Start training from score -1.635400


In [29]:
end_time = Sys.time()
time_matrix[2,]=end_time-start_time

## Target the interval where the average error is minimum

In [30]:
start_time = Sys.time()

In [31]:
dtrain_xg_mean_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_mean)) 
dtrain_xg_mean_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)) )

In [32]:
dtrain_lg_mean_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_mean))
dtrain_lg_mean_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)))

In [33]:
xgb_mean_reg <- xgboost(data = dtrain_xg_mean_reg, nround=100)

[1]	train-rmse:1.496878 
[2]	train-rmse:1.346614 
[3]	train-rmse:1.261811 
[4]	train-rmse:1.214192 
[5]	train-rmse:1.187002 
[6]	train-rmse:1.170034 
[7]	train-rmse:1.157559 
[8]	train-rmse:1.148882 
[9]	train-rmse:1.144276 
[10]	train-rmse:1.138588 
[11]	train-rmse:1.133601 
[12]	train-rmse:1.131450 
[13]	train-rmse:1.128560 
[14]	train-rmse:1.126860 
[15]	train-rmse:1.121205 
[16]	train-rmse:1.118770 
[17]	train-rmse:1.116138 
[18]	train-rmse:1.115413 
[19]	train-rmse:1.114553 
[20]	train-rmse:1.112527 
[21]	train-rmse:1.110023 
[22]	train-rmse:1.106163 
[23]	train-rmse:1.104051 
[24]	train-rmse:1.103132 
[25]	train-rmse:1.100170 
[26]	train-rmse:1.096876 
[27]	train-rmse:1.094057 
[28]	train-rmse:1.092860 
[29]	train-rmse:1.089424 
[30]	train-rmse:1.086710 
[31]	train-rmse:1.085540 
[32]	train-rmse:1.084260 
[33]	train-rmse:1.080781 
[34]	train-rmse:1.078690 
[35]	train-rmse:1.074243 
[36]	train-rmse:1.071490 
[37]	train-rmse:1.070209 
[38]	train-rmse:1.067184 
[39]	train-rmse:1.064

In [34]:
xgb_mean_cl <- xgboost(data = dtrain_xg_mean_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-mlogloss:1.518933 
[2]	train-mlogloss:1.460102 
[3]	train-mlogloss:1.417816 
[4]	train-mlogloss:1.384642 
[5]	train-mlogloss:1.358453 
[6]	train-mlogloss:1.336262 
[7]	train-mlogloss:1.317345 
[8]	train-mlogloss:1.300439 
[9]	train-mlogloss:1.286771 
[10]	train-mlogloss:1.273942 
[11]	train-mlogloss:1.262375 
[12]	train-mlogloss:1.252611 
[13]	train-mlogloss:1.243783 
[14]	train-mlogloss:1.235543 
[15]	train-mlogloss:1.228125 
[16]	train-mlogloss:1.220255 
[17]	train-mlogloss:1.215724 
[18]	train-mlogloss:1.210127 
[19]	train-mlogloss:1.205996 
[20]	train-mlogloss:1.201892 
[21]	train-mlogloss:1.195737 
[22]	train-mlogloss:1.191366 
[23]	train-mlogloss:1.188600 
[24]	train-mlogloss:1.184831 
[25]	train-mlogloss:1.181518 
[26]	train-mlogloss:1.178424 
[27]	train-mlogloss:1.174236 
[28]	train-mlogloss:1.170777 
[29]	train-mlogloss:1.166499 
[30]	train-mlogloss:1.163586 
[31]	train-mlogloss:1.159831 
[32]	train-mlogloss:1.156769 
[33]	train-mlogloss:1.153036 
[34]	train-mlogloss

In [35]:
lgb_mean_reg <- lgb.train(data = dtrain_lg_mean_reg,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012255 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4868
[LightGBM] [Info] Number of data points in the train set: 33791, number of used features: 21
[LightGBM] [Info] Start training from score 1.613921


In [36]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_mean_cl <- lgb.train(data = dtrain_lg_mean_cl,params=params,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014135 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4868
[LightGBM] [Info] Number of data points in the train set: 33791, number of used features: 21
[LightGBM] [Info] Start training from score -1.196925
[LightGBM] [Info] Start training from score -1.644554
[LightGBM] [Info] Start training from score -1.677425
[LightGBM] [Info] Start training from score -1.493495
[LightGBM] [Info] Start training from score -2.371523


In [37]:
end_time = Sys.time()
time_matrix[3,]=end_time-start_time

In [38]:
end_time-start_time

Time difference of 6.743447 mins

## predict

In [39]:
start_time = Sys.time()

In [40]:
alldatalgb <- lgb.Dataset(data = as.matrix(data))
alldataxgb <- xgb.DMatrix(data = as.matrix(data))


In [41]:
xgbregmin=predict(xgb_min_reg,alldataxgb)
xgbclsmin=predict(xgb_min_cl,alldataxgb)
lgbregmin=predict(lgb_min_reg,as.matrix(data))
lgbclsmin=predict(lgb_min_cl,as.matrix(data))

In [42]:
xgbregmin

[1]  2.152967215  2.726136208  3.490833998  0.963540494  1.782772422
    [6]  1.351021767  2.004328489  1.766270757  0.063182786  2.612715960
   [11]  0.940591156  2.551850319  1.824522257  0.314219803  3.474584103
   [16]  1.401627660  2.627771378  1.080519080  1.152341008  1.170283675
   [21]  0.370090187  2.242703199  2.417632580  2.956578255  3.444133997
   [26]  4.245733738  2.937590599  3.557305574  3.121192932  3.017340183
   [31]  2.201222181  3.627236605  2.760938406  3.429369688  3.204774141
   [36]  3.306174040  2.245378256  3.331686497  3.619935036  2.490279675
   [41]  0.604798913  1.714360476  1.608341336  1.581571937  2.643884182
   [46]  0.980133653  1.603292227  1.796469331  3.446124077  1.042351604
   [51]  0.899235666  2.552964926  1.152990341  1.821436882  4.437406540
   [56]  0.811987579  3.739730120  0.875431478  1.632732511  3.270207644
   [61]  2.214380264  1.063173175  2.198515892  1.143623471  0.708917439
   [66]  1.343036532  0.741026700  0.808353841  1.202339053  1.500287652
   [71]  2.345278978  0.895453274  0.396864951  1.192916751  0.559424818
   [76]  1.352250099  2.647163153  1.332638383  2.366533518 -0.063412219
   [81]  1.114540935  1.146957636  1.651744485  1.702612400  1.646305084
   [86]  0.669811845  1.912268877  0.214891940  3.634505987  2.430215120
   [91]  1.969828129  3.558357239  3.873558521  3.108409643  2.699500322
   [96]  1.636866331  2.190882206  1.659093976  2.206583261  2.244879246
  [101]  2.795243263  2.390689850  1.783189535  1.486630917  2.565108299
  [106]  1.397357106  2.767122030  2.576589584  1.549165010  2.412219763
  [111]  1.716117501  2.325438023  2.488443136  4.259027958  2.612607718
  [116]  3.467970133  2.807435513  2.827523708  3.894091129  3.013121367
  [121]  3.848409176  3.580446720  3.222263813  2.951273441  3.042505980
  [126]  3.110945225  1.902128696  3.425821304  2.728754282  3.236620665
  [131]  2.069349766  1.465744972  2.289245844  2.483783007  1.707449079
  [136]  1.517614841  1.200183272  3.610449553  3.604918003  3.152868271
  [141]  1.497338653  3.150889635  1.937798738  3.800989389  3.847596169
  [146]  2.967991114  2.080148458  3.953291416  3.603089333  3.991143942
  [151]  3.995767832  4.504533291  1.533993244  1.023447037  1.725066304
  [156]  3.499234915  3.203178406  2.723104477  3.356287479  1.405358315
  [161]  1.655444264  1.491019845  2.583268642  3.061182976  0.972278595
  [166]  3.605332136  1.042006135  3.336572647  4.503390312  1.054183960
  [171]  3.017807961  3.790796995  1.632302880  3.530216217  2.499430418
  [176]  3.320864201  3.385291576  3.267187357  2.793814182  4.159082413
  [181]  3.388543606  2.623871088  3.855698824  3.078676462  2.728038311
  [186]  3.812362194  3.344994068  3.717531681  2.811560869  3.413211107
  [191]  1.712705374  1.266361713  3.261271954  2.268346786  1.205682635
  [196]  2.863394499  3.217510223  3.482663393  2.704217196  4.507576466
  [201]  1.641276717  4.010640621  2.346408844  2.263463259  0.878168583
  [206]  4.109899998  1.671491861  3.226465702  2.978772879  3.965246916
  [211]  3.763414621  3.576545477  2.747369289  3.127432108  1.158445239
  [216]  1.295568228  1.838637829  1.348262429  1.540421605  1.458715200
  [221]  1.246519089  0.911495030  1.298215389  3.175863981  2.796379328
  [226]  1.363682270  1.298215389  0.967327416  1.429591894  1.553978086
  [231]  0.967327416  2.566266775  2.881872177  3.457875490  2.522015333
  [236]  1.349783182  3.390132427  1.555053473  1.836503267  2.070483685
  [241]  1.037945509  2.288156271  2.816546202  2.181993961  2.992666483
  [246]  2.356641054  0.787754834  1.900164843  2.143348932  3.204990864
  [251]  3.271353960  1.904351830  2.196312904  2.057452440  1.112125278
  [256]  1.722271681  1.751315832  1.270165920  1.129794598  2.771250486
  [261]  1.410220146  1.908277154  1.703421235  1.647153974  2.376046181
  [266]  2.651263237  2.386403799  2.091088295  3.064906359  3.490305662
  [271]  0.966048002  3.620027781  2.929390669  2.692790508  1

In [43]:
xgbclsmin

[1] 4 4 4 0 2 1 3 3 0 3 0 2 4 0 1 0 3 1 1 1 0 3 2 4 3 4 3 4 4 4 3 3 4 3 4 3
   [37] 3 4 3 4 0 1 1 3 4 0 3 0 4 0 0 4 1 0 4 1 4 0 2 4 2 3 3 0 1 0 0 0 2 0 3 0
   [73] 0 2 0 1 3 0 3 0 0 2 0 3 2 0 2 0 4 1 1 4 4 4 2 1 4 1 4 1 4 4 3 0 4 1 3 3
  [109] 3 3 3 3 3 4 4 4 4 3 4 3 4 4 4 4 3 4 3 4 4 4 3 3 3 3 0 1 0 4 4 3 1 4 2 3
  [145] 4 3 3 4 4 4 4 3 0 0 3 3 3 4 4 3 3 3 3 3 0 4 0 3 3 3 3 4 3 3 3 4 3 3 2 4
  [181] 3 4 3 3 4 4 4 4 4 3 2 0 4 3 1 2 4 4 4 3 4 3 3 3 0 3 3 4 4 4 4 3 4 4 0 0
  [217] 3 0 0 0 0 1 0 1 4 0 0 0 2 2 0 3 3 4 4 1 3 3 3 3 1 3 2 1 4 3 1 3 2 3 4 3
  [253] 2 3 0 0 3 0 0 3 1 3 3 3 3 3 2 4 3 3 3 4 4 4 4 3 4 3 0 1 4 3 1 2 4 2 2 2
  [289] 0 3 0 2 3 1 1 2 0 3 0 3 3 0 3 4 4 3 4 2 2 2 0 4 2 0 0 4 3 3 0 3 2 4 0 3
  [325] 1 3 1 4 2 0 0 4 4 4 2 4 0 2 3 4 2 4 4 4 0 0 3 1 0 3 4 4 4 0 3 4 3 4 3 4
  [361] 4 4 0 0 1 1 4 2 3 3 3 3 3 4 4 0 4 3 0 3 4 1 4 0 3 0 4 0 3 4 3 0 3 0 4 3
  [397] 4 3 0 3 2 3 4 4 2 4 0 3 3 0 0 3 3 3 3 4 1 4 4 3 4 4 3 3 4 3 3 3 4 4 3 4
  [433] 4 0 3 3 1 3 0 3 0 4 4 2 4 3 3 0 3 3 4 4 3 3 3 4 3 4 0 3 2 3 3 3 3 3 3 4
  [469] 3 3 3 2 0 3 3 1 1 0 1 0 4 0 2 3 4 2 3 0 4 3 4 3 3 2 4 3 4 4 0 0 3 2 3 2
  [505] 2 4 2 4 3 2 4 3 4 2 3 1 2 4 4 3 3 4 3 3 3 4 2 0 3 3 3 2 2 0 2 4 4 2 2 2
  [541] 3 2 2 2 3 3 4 3 2 3 4 0 4 4 0 0 3 2 0 2 2 3 4 3 1 2 1 3 3 1 1 0 3 3 0 0
  [577] 1 0 0 3 0 4 3 3 4 3 0 3 2 3 0 4 4 2 3 3 1 4 4 4 4 4 3 2 2 3 4 4 3 3 0 3
  [613] 3 4 3 0 4 4 0 4 1 1 1 4 3 3 4 3 4 3 4 3 4 4 4 3 4 3 3 4 4 1 1 0 3 0 3 1
  [649] 2 4 3 4 4 2 4 4 1 4 4 1 2 3 2 4 3 4 4 4 4 3 4 3 3 0 0 4 3 4 3 4 1 4 1 4
  [685] 0 0 1 4 4 0 0 2 4 3 1 3 3 0 3 4 4 4 4 4 4 4 4 3 3 0 3 0 3 4 0 0 2 4 4 4
  [721] 0 2 1 3 2 0 3 1 2 4 1 0 0 4 4 2 0 1 2 1 4 1 1 4 1 4 1 3 3 4 0 4 0 4 1 3
  [757] 0 4 4 2 3 4 4 4 0 0 4 4 4 4 4 1 0 2 4 4 3 4 1 1 1 0 1 0 1 1 4 4 4 1 2 4
  [793] 0 0 4 3 3 3 3 1 1 4 4 4 1 4 1 0 0 0 0 4 4 1 0 1 4 4 4 4 3 4 0 0 3 4 4 4
  [829] 0 4 0 4 2 3 4 3 4 3 4 4 4 4 3 4 3 3 4 4 4 4 3 4 1 4 3 2 4 4 4 4 1 4 4 2
  [865] 4 4 0 1 3 4 1 4 4 4 3 2 1 4 4 1 4 4 4 0 2 4 3 3 1 3 4 2 4 4 4 4 1 3 4 0
  [901] 4 1 4 0 1 4 1 4 4 0 3 4 4 1 0 4 4 0 4 4 4 4 4 3 4 3 0 1 4 1 1 0 4 4 1 0
  [937] 0 4 4 3 4 1 1 0 4 1 4 1 0 4 1 2 2 3 3 1 1 2 0 1 4 4 1 2 4 3 4 2 4 3 0 1
  [973] 0 1 2 1 4 1 4 4 4 4 4 3 4 4 3 4 4 3 2 3 4 4 1 3 3 4 4 3 0 4 4 4 4 4 2 4
 [1009] 1 3 0 1 3 4 1 4 2 4 4 3 4 4 1 0 3 4 0 3 4 3 0 0 4 3 4 1 3 2 3 1 1 2 4 1
 [1045] 2 4 3 1 3 3 3 4 4 0 4 0 2 0 3 1 4 1 2 1 4 4 4 4 4 0 1 0 3 0 4 2 1 3 4 4
 [1081] 1 3 4 4 4 1 2 3 4 0 2 4 4 1 1 3 3 1 4 1 1 2 1 3 4 4 3 3 3 4 3 3 3 3 3 3
 [1117] 3 3 3 1 1 0 3 4 4 3 4 4 3 3 4 0 0 4 4 4 4 1 4 3 4 1 0 3 2 3 4 0 3 4 4 3
 [1153] 4 4 1 3 4 3 2 0 2 2 2 4 3 0 3 1 0 0 4 4 0 0 2 3 3 4 0 4 0 0 0 1 1 4 3 4
 [1189] 3 0 3 3 4 1 0 4 1 3 3 0 1 0 4 4 4 4 4 0 4 4 4 4 2 4 4 3 4 1 0 4 4 4 4 3
 [1225] 4 4 4 4 0 4 1 0 4 4 3 0 4 4 3 1 1 1 0 0 3 4 1 3 4 4 0 0 1 1 4 4 4 3 0 4
 [1261] 4 4 3 0 3 4 4 4 4 3 0 3 3 2 4 3 3 4 4 4 4 3 3 1 1 4 4 4 4 3 3 2 3 4 4 3
 [1297] 4 4 1 2 4 4 4 3 1 4 4 3 4 0 0 4 1 4 1 4 4 4 4 3 4 4 1 4 4 3 2 4 0 4 4 4
 [1333] 4 4 2 4 4 3 3 0 4 3 0 4 4 3 4 0 4 4 4 0 4 4 3 4 3 4 4 0 3 3 4 2 4 4 0 2
 [1369] 1 4 4 3 4 3 0 4 0 4 0 4 4 4 4 4 4 4 0 0 3 4 4 3 4 3 4 4 3 0 1 4 4 4 4 3
 [1405] 0 3 4 4 4 4 3 1 0 3 3 4 3 4 0 4 2 0 2 0 3 4 0 1 3 0 3 1 4 0 3 2 4 1 4 4
 [1441] 4 3 2 0 4 3 1 4 3 3 4 4 3 4 4 2 4 4 3 0 4 4 3 4 0 2 1 4 4 4 1 1 4 4 4 4
 [1477] 4 0 1 3 4 1 4 0 2 4 4 0 0 3 3 0 0 1 3 0 0 0 4 4 4 4 0 4 4 4 0 4 4 2 4 0
 [1513] 3 0 2 4 0 4 4 3 0 4 2 2 0 4 3 1 0 2 4 4 3 0 3 3 3 3 4 1 3 4 0 1 4 4 4 1
 [1549] 0 0 0 4 3 4 4 0 0 3 3 3 4 4 4 4 1 3 0 4 4 3 4 4 2 0 3 3 3 4 4 4 3 4 4 4
 [1585] 0 1 1 3 0 1 3 4 4 0 4 1 3 4 4 4 3 4 3 4 0 4 4 4 4 4 4 0 0 4 4 4 3 0 1 0
 [1621] 4 0 1 4 4 1 0 3 4 0 4 3 3 1 4 1 0 3 4 0 4 2 3 0 1 4 4 3 4 3 4 4 3 1 0 4
 [1657] 4 1 1 4 4 4 4 4 2 3 3 4 4 4 4 0 3 0 1 4 4 4 4 4 0 0 2 0 0 4 4 3 3 2 1 0
 [1693] 1 4 0 0 1 2 4 4 4 4 0 4 1 4 4 4 3 4 3 0 3 4 4 4 1 4 1 4 4 4 1 1 4 4 4 4
 [1729] 2 4 3 3 2 4 4 0 0 3 0 0 4 0 4 2 4 3 4 4 4 4 4 4 2 1 0 4 4 4 4 4 0 3 4 0
 [1765] 4 4 4 3 4 4 4 4 0 0 3 4 0 2 4 3 1 4 4 3 3 4 1 4 1 4 1 4 0 1 4 3 4 1 1 0
 [18

In [44]:
lgbregmin

[1] 2.4162667 2.4605042 3.2152334 1.3042642 1.7996480 1.5336135 1.8292733
    [8] 1.8955540 1.8232382 2.1242919 1.8541989 1.3535123 2.0869732 1.9481133
   [15] 2.7368334 1.9542744 2.5212819 2.2616556 1.5393794 1.8791604 1.6278143
   [22] 1.9633285 1.7394255 3.4481567 3.4276742 3.8710147 3.3992814 2.7402018
   [29] 3.2327751 3.0967395 2.1389004 3.1067284 3.2502589 2.8050457 2.7978936
   [36] 3.0729307 2.2622333 3.3016784 2.8224461 2.6140818 1.3373944 1.3172042
   [43] 1.4198436 2.3304740 2.0126956 1.9908029 1.9430552 1.7536393 2.2965934
   [50] 1.4498059 0.9341517 3.1245654 1.5630590 1.5473349 3.5718051 1.5080536
   [57] 2.5823813 0.7685219 1.3321480 2.3461712 1.8308320 1.7653867 1.8659418
   [64] 1.3075095 0.8799400 1.3480145 0.7968576 1.4883606 1.5411848 1.5610258
   [71] 2.1496536 0.7038356 1.3992019 1.3789446 1.3018129 1.8023290 1.9999078
   [78] 0.9426251 1.5371225 1.5563162 0.8125816 1.3744360 1.0073777 1.4855452
   [85] 1.4833113 1.7683391 1.3871345 1.1875033 2.0828489 1.7824946 1.9631064
   [92] 3.2154588 3.7828526 2.6039288 2.4563091 1.7386971 2.1870739 1.8910747
   [99] 2.0673534 1.8264036 2.7254626 2.4237855 1.7500231 1.5446098 3.0523361
  [106] 1.4143668 2.0556834 1.6886095 1.7128345 2.2825687 1.4686441 2.2208121
  [113] 2.2214702 3.9076745 2.4869317 3.1091846 3.1295576 3.4175094 3.7869050
  [120] 3.6542312 3.6858009 3.5829072 3.3346618 2.6157281 2.4316785 3.0260550
  [127] 3.2679366 2.8888663 2.9684904 3.3862374 1.7217022 1.3069685 1.7576847
  [134] 1.8703263 1.3194096 1.5414046 1.4325492 3.2233642 3.2641308 3.2500663
  [141] 1.6075003 3.2345651 1.7657195 3.0299092 3.7409628 2.0936779 1.4708057
  [148] 3.6567164 3.4613281 3.8676832 3.6189715 3.2596960 1.1753437 1.1753437
  [155] 1.5134600 3.1395220 3.3641504 3.5179459 3.5401392 1.6030586 1.6462588
  [162] 1.6124222 2.6965384 2.7946263 1.5933348 3.5575716 0.7744563 3.1800017
  [169] 3.4288681 1.4641698 2.7105783 3.3919694 1.6721795 2.9674907 3.0649326
  [176] 3.2941832 3.1192448 3.2946079 2.8680745 3.4840896 3.1965272 1.9640176
  [183] 3.2369120 3.1099365 2.5487376 3.7708644 3.4200090 3.6913049 2.9649117
  [190] 2.8999182 1.4178494 1.4197558 3.5779439 1.5563269 1.4860405 3.4326821
  [197] 3.4235239 3.3419103 2.7117940 3.0928953 2.0028298 3.1319957 2.1588623
  [204] 1.7640986 1.3684838 3.3563842 1.2676024 3.1888126 3.0912898 3.6364112
  [211] 3.7511535 3.0861529 2.6326416 3.0540809 1.1889128 1.2385994 1.3514146
  [218] 0.9003609 1.4182468 1.1713724 1.0331287 1.2316232 1.6513673 2.5936608
  [225] 2.5819551 1.5749118 1.6513673 1.5175458 1.2093749 1.7791229 1.5175458
  [232] 2.3117770 2.9029277 3.4019048 2.3661023 1.7756538 3.3432192 1.4710009
  [239] 1.5438527 1.5608694 1.6057404 1.8577413 2.8122171 1.7753454 3.2387922
  [246] 1.7372624 1.3778286 1.6517732 1.9094608 3.2375720 3.1716002 1.4621917
  [253] 1.9471466 1.9561425 1.3252670 1.2675116 1.6677875 1.4005225 1.3022752
  [260] 3.1023666 1.6634994 1.7787046 1.6878160 1.2118409 2.1765393 2.0665254
  [267] 2.7718173 2.7742166 2.4761675 2.5185175 1.5872831 2.7090416 3.0200213
  [274] 2.5631836 2.3018550 2.7464819 2.2711385 1.2565058 1.8420684 1.7317441
  [281] 2.2196710 1.7720683 0.9957958 1.6526117 2.5406377 1.6577020 1.5061925
  [288] 2.9499447 1.4339449 1.9973946 2.0301112 1.9814312 1.9232372 1.8783249
  [295] 1.0112795 1.6616308 1.0632933 1.4462226 2.0070447 2.3684228 2.1437762
  [302] 1.5845739 1.6205557 3.2578955 2.7448271 2.4704523 3.4102753 3.4964332
  [309] 3.3029620 2.2243318 1.9568455 2.0572579 1.5365967 1.1949015 1.4362051
  [316] 3.4296034 1.5049041 1.4771182 2.4970227 1.8772296 3.0064348 2.4268628
  [323] 1.4209849 1.9370829 1.9612369 1.9164010 1.4882414 3.1413203 3.4002007
  [330] 1.7342495 1.6585279 2.8017221 3.0211325 2.8370836 3.1979117 2.9711017
  [337] 1.6134444 2.2594198 2.8840058 2.4275019 1.3723743 3.2993221 3.1152401
  [344] 2.7587260 1.2203978 1.4875976 2.8274441 1.5179481 1.6811473 2.1533442
  [351] 3.2883788 3.0701452 3.5552745 1.5763014 2.6505733 2.9039240 2.3025766
  [358] 2.6165484 2.539338

In [45]:
lgbclsmin

0.28316338,0.06331474,0.03993616,0.1051749,5.084109e-01
0.18668758,0.14071223,0.06239763,0.1209547,4.892479e-01
0.02679570,0.01657602,0.05014666,0.3201268,5.863548e-01
0.39908533,0.13171819,0.18882350,0.2802838,8.921875e-05
0.20173083,0.21064295,0.25384562,0.3336585,1.221310e-04
0.16762197,0.31510448,0.10361885,0.1220277,2.916270e-01
0.13581869,0.16422076,0.19545143,0.5043539,1.552365e-04
0.15994945,0.17252774,0.31031025,0.3570498,1.628031e-04
0.46834242,0.05785328,0.08316034,0.1009863,2.896577e-01
0.26118103,0.16869120,0.06111416,0.3103215,1.986921e-01
0.32851642,0.15265178,0.09213153,0.1970541,2.296461e-01


In [46]:
datalength=dim(data)[1]
lgbclsm=matrix(lgbclsmin,m,datalength)
lgbclsminr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsminr[i,]=which.max(lgbclsm[,i])
}
lgbclsminr=lgbclsminr-1

In [47]:
lgbclsminr

3
3
0
0
0
4
0
1
0
0
0


In [48]:
xgbregmean=predict(xgb_mean_reg,alldataxgb)
xgbclsmean=predict(xgb_mean_cl,alldataxgb)
lgbregmean=predict(lgb_mean_reg,as.matrix(data))
lgbclsmean=predict(lgb_mean_cl,as.matrix(data))

In [49]:
lgbclsm=matrix(lgbclsmean,m,datalength)
lgbclsmeanr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsmeanr[i,]=which.max(lgbclsm[,i])
}
lgbclsmeanr=lgbclsmeanr-1

In [50]:
preallmin=cbind(xgbclsmin,xgbregmin)
preallmin=cbind(preallmin,lgbclsminr)
preallmin=cbind(preallmin,lgbregmin)

In [51]:
colnames(preallmin)=c('xgbclsmin','xgbregmin','lgbclsmin','lgbregmin')
head(preallmin)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
4,2.1529672,3,2.416267
4,2.7261362,3,2.460504
4,3.4908340,0,3.215233
0,0.9635405,0,1.304264
2,1.7827724,0,1.799648
1,1.3510218,4,1.533613


In [52]:
preallmean=cbind(xgbclsmean,xgbregmean)
preallmean=cbind(preallmean,lgbclsmeanr)
preallmean=cbind(preallmean,lgbregmean)

In [53]:
head(preallmean)

xgbclsmean,xgbregmean,,lgbregmean
3,2.7869437,3,1.9393675
3,2.2656732,3,2.2951503
3,3.0697255,0,2.7336085
1,0.9555054,2,0.8140261
3,1.4852703,0,1.6747099
1,1.2191916,2,1.7768961


In [54]:
colnames(preallmean)=c('xgbclsmean','xgbregmean','lgbclsmean','lgbregmean')
head(preallmean)

xgbclsmean,xgbregmean,lgbclsmean,lgbregmean
3,2.7869437,3,1.9393675
3,2.2656732,3,2.2951503
3,3.0697255,0,2.7336085
1,0.9555054,2,0.8140261
3,1.4852703,0,1.6747099
1,1.2191916,2,1.7768961


In [55]:
end_time = Sys.time()
time_matrix[4,]=end_time-start_time

In [56]:
result_list=list(preallmin,preallmean,time_matrix)

In [57]:
save(result_list, file = "Monthly_nnetar_opt_pre_result.RData")

In [58]:
time_matrix

user_time,system_time,elapsed_time
19.287469,19.287469,19.287469
6.736094,6.736094,6.736094
6.743447,6.743447,6.743447
2.278507,2.278507,2.278507
